# 02.4 - CNN gene_group voi sinusoidal positional encoding

Notebook nay thu cach them vi tri vao CNN bang positional encoding co dinh theo vi tri trong window 201bp.

Input goc `201 x 9` se duoc ghep them 8 kenh sin/cos positional encoding, thanh `201 x 17`. Split van la `gene_group` de so sanh cong bang voi notebook 02.3.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"

X_PATH = PROCESSED_DIR / "X_ref_alt_marker_201.npy"
Y_PATH = PROCESSED_DIR / "y.npy"
METADATA_PATH = PROCESSED_DIR / "clinvar_training_metadata.parquet"

for path in [X_PATH, Y_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Chua co file: {path}. Hay chay notebook 01 den buoc 12B sau khi them tensor ref/alt/marker."
        )

X_PATH, Y_PATH, METADATA_PATH

(PosixPath('/mnt/MyData/Bioinformatics/Project/processed/X_ref_alt_marker_201.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/y.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_training_metadata.parquet'))

## 1. Load dataset da preprocess

`X` co shape `(n_variants, 201, 9)`.

9 channels gom:

- `0:4`: one-hot `ref_seq` theo thu tu A/C/G/T
- `4:8`: one-hot `alt_seq` theo thu tu A/C/G/T
- `8`: mutation marker, bang 1 tai vi tri center index 100

PyTorch `Conv1d` can input `(batch, channels, length)`, nen trong `Dataset` se transpose thanh `(9, 201)`.

In [2]:
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata_df = pd.read_parquet(METADATA_PATH)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"metadata shape: {metadata_df.shape}")
print(f"X dtype: {X.dtype}")
print(f"y dtype: {y.dtype}")

assert X.ndim == 3
assert X.shape[1:] == (201, 9)
assert len(X) == len(y) == len(metadata_df)

pd.Series(y).value_counts().rename(index={0: "Benign/Likely benign", 1: "Pathogenic/Likely pathogenic"})

X shape: (460488, 201, 9)
y shape: (460488,)
metadata shape: (460488, 15)
X dtype: uint8
y dtype: int8


Benign/Likely benign            401054
Pathogenic/Likely pathogenic     59434
Name: count, dtype: int64

## 2. Tao train/validation/test split theo gene

Khac notebook 02 random theo row, notebook nay dung `GroupShuffleSplit` theo `GeneSymbol`. Cac gene `unknown` neu co se duoc tach thanh group rieng theo row de khong gom mot group qua lon.


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

RANDOM_STATE = 42
SPLIT_MODE = "gene_group"

# De None de train full. Doi thanh so nho, vi du 20_000, neu chi muon smoke test pipeline.
DEBUG_MAX_TRAIN_ROWS = None
DEBUG_MAX_VAL_ROWS = None
DEBUG_MAX_TEST_ROWS = None

indices = np.arange(len(y))
groups = metadata_df["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
unknown_mask = groups == "unknown"
groups[unknown_mask] = np.array([f"unknown_{i}" for i in np.where(unknown_mask)[0]])

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(indices, y, groups=groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
rel_val_idx, rel_test_idx = next(gss2.split(temp_idx, y[temp_idx], groups=groups[temp_idx]))
val_idx = temp_idx[rel_val_idx]
test_idx = temp_idx[rel_test_idx]


def maybe_subsample(idx, max_rows, name):
    if max_rows is None or len(idx) <= max_rows:
        return idx
    sampled, _ = train_test_split(
        idx,
        train_size=max_rows,
        random_state=RANDOM_STATE,
        stratify=y[idx],
    )
    print(f"DEBUG: {name} subset {len(idx):,} -> {len(sampled):,}")
    return sampled

train_idx = maybe_subsample(train_idx, DEBUG_MAX_TRAIN_ROWS, "train")
val_idx = maybe_subsample(val_idx, DEBUG_MAX_VAL_ROWS, "val")
test_idx = maybe_subsample(test_idx, DEBUG_MAX_TEST_ROWS, "test")

print("SPLIT_MODE:", SPLIT_MODE)
print(f"train: {len(train_idx):,}, labels={dict(zip(*np.unique(y[train_idx], return_counts=True)))}")
print(f"val:   {len(val_idx):,}, labels={dict(zip(*np.unique(y[val_idx], return_counts=True)))}")
print(f"test:  {len(test_idx):,}, labels={dict(zip(*np.unique(y[test_idx], return_counts=True)))}")

train_genes = set(metadata_df.iloc[train_idx]["GeneSymbol"].fillna("unknown").astype(str)) - {"unknown"}
val_genes = set(metadata_df.iloc[val_idx]["GeneSymbol"].fillna("unknown").astype(str)) - {"unknown"}
test_genes = set(metadata_df.iloc[test_idx]["GeneSymbol"].fillna("unknown").astype(str)) - {"unknown"}
print("gene overlap train/val:", len(train_genes & val_genes))
print("gene overlap train/test:", len(train_genes & test_genes))
print("gene overlap val/test:", len(val_genes & test_genes))

key_cols = ["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
def variant_keys(idx):
    return set(map(tuple, metadata_df.iloc[idx][key_cols].astype(str).to_numpy()))
print("variant key overlap train/val:", len(variant_keys(train_idx) & variant_keys(val_idx)))
print("variant key overlap train/test:", len(variant_keys(train_idx) & variant_keys(test_idx)))


SPLIT_MODE: gene_group
train: 319,608, labels={np.int8(0): np.int64(278577), np.int8(1): np.int64(41031)}
val:   72,308, labels={np.int8(0): np.int64(62401), np.int8(1): np.int64(9907)}
test:  68,572, labels={np.int8(0): np.int64(60076), np.int8(1): np.int64(8496)}
gene overlap train/val: 0
gene overlap train/test: 0
gene overlap val/test: 0
variant key overlap train/val: 0
variant key overlap train/test: 0


## 2B. Audit khoang cach genome train/test

Gene split khong dam bao genomic distance xa tuyet doi, nhung se loai bo shortcut cung gene. Cell nay do nhanh nearest train distance tren cung chromosome.


In [4]:
train_pos_by_chr = {}
for chrom, sub in metadata_df.iloc[train_idx].groupby("Chromosome", sort=False):
    train_pos_by_chr[str(chrom)] = np.sort(
        pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna().astype(np.int64).to_numpy()
    )


def nearest_train_distances(idx):
    chunks = []
    for chrom, sub in metadata_df.iloc[idx].groupby("Chromosome", sort=False):
        arr = train_pos_by_chr.get(str(chrom))
        if arr is None or len(arr) == 0:
            continue
        pos = pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna().astype(np.int64).to_numpy()
        j = np.searchsorted(arr, pos)
        d = np.full(len(pos), np.iinfo(np.int64).max, dtype=np.int64)
        m = j < len(arr)
        d[m] = np.minimum(d[m], np.abs(arr[j[m]] - pos[m]))
        m = j > 0
        d[m] = np.minimum(d[m], np.abs(arr[j[m] - 1] - pos[m]))
        chunks.append(d)
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.int64)

for split_name, idx in [("val", val_idx), ("test", test_idx)]:
    d = nearest_train_distances(idx)
    print("\n", split_name)
    print("quantiles:", {q: int(np.quantile(d, q)) for q in [0, .01, .05, .10, .25, .50, .75, .90, .95, .99]})
    for t in [1, 10, 50, 100, 201, 500, 1000, 10000]:
        print(f"pct <= {t:5,d} bp: {(d <= t).mean():.4f}")



 val
quantiles: {0: 0, 0.01: 121, 0.05: 2271, 0.1: 6396, 0.25: 16440, 0.5: 36009, 0.75: 76095, 0.9: 168264, 0.95: 216252, 0.99: 567150}
pct <=     1 bp: 0.0005
pct <=    10 bp: 0.0022
pct <=    50 bp: 0.0067
pct <=   100 bp: 0.0091
pct <=   201 bp: 0.0122
pct <=   500 bp: 0.0190
pct <= 1,000 bp: 0.0265
pct <= 10,000 bp: 0.1505

 test
quantiles: {0: 0, 0.01: 209, 0.05: 3649, 0.1: 6751, 0.25: 17656, 0.5: 38932, 0.75: 112705, 0.9: 241188, 0.95: 405967, 0.99: 849232}
pct <=     1 bp: 0.0005
pct <=    10 bp: 0.0017
pct <=    50 bp: 0.0048
pct <=   100 bp: 0.0069
pct <=   201 bp: 0.0098
pct <=   500 bp: 0.0150
pct <= 1,000 bp: 0.0199
pct <= 10,000 bp: 0.1513


## 3. Tao PyTorch Dataset/DataLoader voi positional encoding

Moi sample lay `X[idx]` dang `(201, 9)`, sau do concat them positional encoding `(201, 8)` theo truc channel. Conv1D nhan input `(17, 201)`.


In [5]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Positional encoding config.
POSITIONAL_ENCODING_DIM = 8
INPUT_CHANNELS = X.shape[2] + POSITIONAL_ENCODING_DIM

# Imbalance handling.
USE_WEIGHTED_SAMPLER = True
POS_WEIGHT_MODE = "sqrt"  # "none", "sqrt", hoac "full"

# Hard-negative fine-tuning.
USE_HARD_NEGATIVE_FINETUNE = True
HARD_NEGATIVE_THRESHOLD = 0.50
HARD_NEGATIVE_EPOCHS = 2
HARD_NEGATIVE_LR = 3e-4
HARD_NEGATIVE_MAX_EASY_NEG_RATIO = 1.0


def make_sinusoidal_position_encoding(seq_len, dim):
    if dim % 2 != 0:
        raise ValueError("POSITIONAL_ENCODING_DIM phai la so chan")
    positions = np.arange(seq_len, dtype=np.float32)[:, None]
    div_term = np.exp(np.arange(0, dim, 2, dtype=np.float32) * (-np.log(10000.0) / dim))
    pe = np.zeros((seq_len, dim), dtype=np.float32)
    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe


POSITIONAL_ENCODING = make_sinusoidal_position_encoding(X.shape[1], POSITIONAL_ENCODING_DIM)
print("POSITIONAL_ENCODING shape:", POSITIONAL_ENCODING.shape)
print("INPUT_CHANNELS:", INPUT_CHANNELS)


class ClinVarSequenceDataset(Dataset):
    def __init__(self, X_array, y_array, indices, positional_encoding):
        self.X_array = X_array
        self.y_array = y_array
        self.indices = np.asarray(indices)
        self.positional_encoding = positional_encoding.astype(np.float32, copy=False)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = self.indices[item]
        x_base = np.asarray(self.X_array[idx], dtype=np.float32)
        x_np = np.concatenate([x_base, self.positional_encoding], axis=1).T.copy()
        x = torch.from_numpy(x_np).float()
        target = torch.tensor(self.y_array[idx], dtype=torch.float32)
        return x, target


def make_weighted_sampler(indices):
    labels = y[np.asarray(indices)]
    class_count = np.bincount(labels, minlength=2)
    class_weight = 1.0 / np.maximum(class_count, 1)
    sample_weight = class_weight[labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weight),
        num_samples=len(indices),
        replacement=True,
    )


BATCH_SIZE = 512

train_dataset = ClinVarSequenceDataset(X, y, train_idx, POSITIONAL_ENCODING)
train_sampler = make_weighted_sampler(train_idx) if USE_WEIGHTED_SAMPLER else None

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    ClinVarSequenceDataset(X, y, val_idx, POSITIONAL_ENCODING),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    ClinVarSequenceDataset(X, y, test_idx, POSITIONAL_ENCODING),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape, batch_y.shape)
print("USE_WEIGHTED_SAMPLER:", USE_WEIGHTED_SAMPLER)
print("POS_WEIGHT_MODE:", POS_WEIGHT_MODE)
print("USE_HARD_NEGATIVE_FINETUNE:", USE_HARD_NEGATIVE_FINETUNE)


Device: cuda
POSITIONAL_ENCODING shape: (201, 8)
INPUT_CHANNELS: 17
torch.Size([512, 17, 201]) torch.Size([512])
USE_WEIGHTED_SAMPLER: True
POS_WEIGHT_MODE: sqrt
USE_HARD_NEGATIVE_FINETUNE: True


## 4. Build CNN 1D PyTorch

Khac notebook 02.3 o `Conv1d(INPUT_CHANNELS, 64, ...)`, vi input co them 8 kenh positional encoding.


In [6]:
class ClinVarCNN(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x).squeeze(1)


model = ClinVarCNN(INPUT_CHANNELS).to(DEVICE)
model


ClinVarCNN(
  (features): Sequential(
    (0): Conv1d(17, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveMaxPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.2, inplace=F

## 5. Train nhanh

Dung `BCEWithLogitsLoss`. Neu class imbalance, `pos_weight` se tang/giam trong loss cho class `1`.

In [7]:
# PyTorch/sympy compatibility guard.
# In some long-running notebook kernels, sympy.core is not attached to the sympy module
# when torch.optim lazily imports torch._dynamo. Force-load it before creating optimizer.
import importlib
import sympy

if not hasattr(sympy, "core"):
    sympy.core = importlib.import_module("sympy.core")
if not hasattr(sympy.core, "symbol"):
    sympy.core.symbol = importlib.import_module("sympy.core.symbol")

positive_count = (y_train := y[train_idx]).sum()
negative_count = len(y_train) - positive_count
full_pos_weight = negative_count / max(positive_count, 1)

if POS_WEIGHT_MODE == "none":
    pos_weight_value = 1.0
elif POS_WEIGHT_MODE == "sqrt":
    pos_weight_value = float(np.sqrt(full_pos_weight))
elif POS_WEIGHT_MODE == "full":
    pos_weight_value = float(full_pos_weight)
else:
    raise ValueError(f"POS_WEIGHT_MODE khong hop le: {POS_WEIGHT_MODE}")

pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"full_pos_weight: {full_pos_weight:.4f}")
print(f"used_pos_weight: {pos_weight.item():.4f}")


full_pos_weight: 6.7894
used_pos_weight: 2.6057


In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score


def run_epoch(model, loader, train=False, epoch_label=""):
    model.train(train)
    total_loss = 0.0
    all_targets = []
    all_probs = []
    mode = "train" if train else "eval"

    progress = tqdm(loader, desc=f"{mode} {epoch_label}".strip(), unit="batch", leave=False)
    for batch_x, batch_y in progress:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(batch_y.detach().cpu().numpy())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    targets = np.concatenate(all_targets)
    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(np.int8)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": (preds == targets).mean(),
        "roc_auc": roc_auc_score(targets, probs),
        "pr_auc": average_precision_score(targets, probs),
    }
    return metrics, probs


EPOCHS = 5
best_val_auc = -np.inf
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_metrics, _ = run_epoch(model, train_loader, train=True, epoch_label=f"main {epoch}/{EPOCHS}")
    val_metrics, _ = run_epoch(model, val_loader, train=False, epoch_label=f"main {epoch}/{EPOCHS}")

    row = {"phase": "main", "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

    if val_metrics["pr_auc"] > best_val_auc:
        best_val_auc = val_metrics["pr_auc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"New best val_pr_auc: {best_val_auc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

if USE_HARD_NEGATIVE_FINETUNE:
    print()
    print("Hard-negative mining on train set...")
    _, train_probs_ordered = run_epoch(model, train_eval_loader, train=False, epoch_label="mine train")
    train_labels_ordered = y[train_idx]
    hard_neg_mask = (train_labels_ordered == 0) & (train_probs_ordered >= HARD_NEGATIVE_THRESHOLD)
    pos_idx = train_idx[train_labels_ordered == 1]
    hard_neg_idx = train_idx[hard_neg_mask]
    easy_neg_idx = train_idx[(train_labels_ordered == 0) & ~hard_neg_mask]

    rng = np.random.default_rng(RANDOM_STATE)
    max_easy = int((len(pos_idx) + len(hard_neg_idx)) * HARD_NEGATIVE_MAX_EASY_NEG_RATIO)
    if len(easy_neg_idx) > max_easy:
        easy_neg_idx = rng.choice(easy_neg_idx, size=max_easy, replace=False)

    hard_train_idx = np.concatenate([pos_idx, hard_neg_idx, easy_neg_idx])
    rng.shuffle(hard_train_idx)

    print(f"positive train rows: {len(pos_idx):,}")
    print(f"hard negative rows:  {len(hard_neg_idx):,}")
    print(f"easy negative rows:  {len(easy_neg_idx):,}")
    print(f"fine-tune rows:      {len(hard_train_idx):,}")

    hard_loader = DataLoader(
        ClinVarSequenceDataset(X, y, hard_train_idx, POSITIONAL_ENCODING),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=HARD_NEGATIVE_LR, weight_decay=1e-4)
    for epoch in range(1, HARD_NEGATIVE_EPOCHS + 1):
        print(f"Hard-negative epoch {epoch}/{HARD_NEGATIVE_EPOCHS}")
        train_metrics, _ = run_epoch(model, hard_loader, train=True, epoch_label=f"hard {epoch}/{HARD_NEGATIVE_EPOCHS}")
        val_metrics, _ = run_epoch(model, val_loader, train=False, epoch_label=f"hard {epoch}/{HARD_NEGATIVE_EPOCHS}")
        row = {"phase": "hard_negative", "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
        history.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

        if val_metrics["pr_auc"] > best_val_auc:
            best_val_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"New best val_pr_auc: {best_val_auc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df


Epoch 1/5


train main 1/5:   0%|          | 0/625 [00:00<?, ?batch/s]

eval main 1/5:   0%|          | 0/142 [00:00<?, ?batch/s]

{'phase': 'main', 'epoch': 1, 'train_loss': 0.8795, 'train_accuracy': np.float64(0.6409), 'train_roc_auc': 0.7723, 'train_pr_auc': 0.7637, 'val_loss': 0.715, 'val_accuracy': np.float64(0.6608), 'val_roc_auc': 0.8373, 'val_pr_auc': 0.4906}
New best val_pr_auc: 0.4906
Epoch 2/5


train main 2/5:   0%|          | 0/625 [00:00<?, ?batch/s]

...

{'phase': 'hard_negative', 'epoch': 1, 'train_loss': 0.4508, 'train_accuracy': np.float64(0.847), 'train_roc_auc': 0.8968, 'train_pr_auc': 0.5943, 'val_loss': 0.5293, 'val_accuracy': np.float64(0.8729), 'val_roc_auc': 0.8621, 'val_pr_auc': 0.5507}
New best val_pr_auc: 0.5507
Hard-negative epoch 2/2


train hard 2/2:   0%|          | 0/563 [00:00<?, ?batch/s]

eval hard 2/2:   0%|          | 0/142 [00:00<?, ?batch/s]

{'phase': 'hard_negative', 'epoch': 2, 'train_loss': 0.4316, 'train_accuracy': np.float64(0.8562), 'train_roc_auc': 0.9058, 'train_pr_auc': 0.6176, 'val_loss': 0.5453, 'val_accuracy': np.float64(0.8738), 'val_roc_auc': 0.8643, 'val_pr_auc': 0.5555}
New best val_pr_auc: 0.5555


,phase,epoch,train_loss,train_accuracy,train_roc_auc,train_pr_auc,val_loss,val_accuracy,val_roc_auc,val_pr_auc
0,main,1,0.879511,0.640913,0.772316,0.763708,0.714985,0.660798,0.837269,0.490594
1,main,2,0.743191,0.730974,0.848680,0.835169,0.614063,0.730777,0.849914,0.519609
2,main,3,0.685345,0.759909,0.871617,0.856438,0.585752,0.750097,0.854235,0.523101
3,main,4,0.641390,0.778635,0.885886,0.869477,0.594591,0.739088,0.857348,0.534239
4,main,5,0.602017,0.795715,0.898629,0.882161,0.720822,0.667188,0.859612,0.539228
5,hard_negative,1,0.450809,0.846992,0.896777,0.594345,0.529305,0.872932,0.862130,0.550716
6,hard_negative,2,0.431567,0.856230,0.905801,0.617594,0.545285,0.873790,0.864300,0.555504


## 6. Danh gia tren test set

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

test_metrics, proba_test = run_epoch(model, test_loader, train=False, epoch_label="test")
pred_test = (proba_test >= 0.5).astype(np.int8)
y_test = y[test_idx]

print(test_metrics)
print("Classification report:")
print(classification_report(y_test, pred_test, target_names=["benign", "pathogenic"], zero_division=0))

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=["true_benign", "true_pathogenic"],
    columns=["pred_benign", "pred_pathogenic"],
)
confusion_matrix_df


eval test:   0%|          | 0/134 [00:00<?, ?batch/s]

{'loss': 0.4701236963014804, 'accuracy': np.float64(0.8835384705127457), 'roc_auc': 0.8806323333037412, 'pr_auc': 0.5722289219735142}
Classification report:
              precision    recall  f1-score   support

      benign       0.94      0.93      0.93     60076
  pathogenic       0.53      0.58      0.55      8496

    accuracy                           0.88     68572
   macro avg       0.73      0.75      0.74     68572
weighted avg       0.89      0.88      0.89     68572



,pred_benign,pred_pathogenic
true_benign,55649,4427
true_pathogenic,3559,4937


## 6A. Threshold table cho precision/recall

Thay vi dung co dinh threshold `0.5`, cell nay tinh bang threshold de chon operating point: uu tien recall pathogenic hay precision pathogenic.


In [10]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, proba_test)
rows = []
for threshold, p_value, r_value in zip(thresholds, precision[:-1], recall[:-1]):
    f1 = 2 * p_value * r_value / (p_value + r_value) if (p_value + r_value) else 0.0
    beta = 2
    f2 = (1 + beta**2) * p_value * r_value / (beta**2 * p_value + r_value) if (beta**2 * p_value + r_value) else 0.0
    rows.append({
        "threshold": float(threshold),
        "precision": float(p_value),
        "recall": float(r_value),
        "f1": float(f1),
        "f2": float(f2),
    })

threshold_df = pd.DataFrame(rows)

for target_precision in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    candidates = threshold_df[threshold_df["precision"] >= target_precision]
    best_recall = candidates["recall"].max() if len(candidates) else 0.0
    print(f"Recall at precision >= {target_precision:.2f}: {best_recall:.4f}")

for target_recall in [0.50, 0.60, 0.70, 0.80, 0.90]:
    candidates = threshold_df[threshold_df["recall"] >= target_recall]
    best_precision = candidates["precision"].max() if len(candidates) else 0.0
    print(f"Precision at recall >= {target_recall:.2f}: {best_precision:.4f}")

print("Best F1 threshold:")
display(threshold_df.sort_values("f1", ascending=False).head(1))
print("Best F2 threshold:")
display(threshold_df.sort_values("f2", ascending=False).head(1))

display(threshold_df.iloc[np.linspace(0, len(threshold_df) - 1, 12, dtype=int)])


Recall at precision >= 0.30: 0.8723
Recall at precision >= 0.40: 0.7479
Recall at precision >= 0.50: 0.6199
Recall at precision >= 0.60: 0.4657
Recall at precision >= 0.70: 0.3186
Recall at precision >= 0.80: 0.1729
Precision at recall >= 0.50: 0.5768
Precision at recall >= 0.60: 0.5134
Precision at recall >= 0.70: 0.4368
Precision at recall >= 0.80: 0.3579
Precision at recall >= 0.90: 0.2737
Best F1 threshold:


,threshold,precision,recall,f1,f2
58021,0.468457,0.502007,0.618173,0.554067,0.590829


Best F2 threshold:


,threshold,precision,recall,f1,f2
50482,0.273909,0.371148,0.786723,0.504358,0.642779


,threshold,precision,recall,f1,f2
0,0.000001,0.123899,1.000000,0.220481,0.414213
6223,0.000971,0.135951,0.997646,0.239293,0.439946
12447,0.002760,0.150327,0.992820,0.261117,0.468117
18671,0.006162,0.167679,0.984228,0.286542,0.498611
24895,0.012568,0.189146,0.971398,0.316638,0.531649
31119,0.025733,0.216416,0.952448,0.352692,0.566865
37343,0.055011,0.251597,0.922787,0.395390,0.601735
43567,0.123488,0.298011,0.874529,0.444538,0.630559
49791,0.255812,0.361371,0.795433,0.496967,0.641359
56015,0.413574,0.458861,0.673493,0.545836,0.615878


## 6B. Cach doc ket qua gene_group split

Ket qua o notebook nay se thuong thap hon notebook 02 random split. Do la dieu binh thuong va dang mong doi, vi test set gom cac gene khong xuat hien trong train.

Nen so sanh cac chi so sau voi notebook 02:

- `PR-AUC`: quan trong nhat cho class pathogenic thua so.
- `ROC-AUC`: do kha nang xep hang tong quat.
- pathogenic precision/recall trong classification report.
- confusion matrix, dac biet false negative pathogenic.

Neu gene_group split tut manh, ket qua random split truoc do dang optimistic do cung gene/cung neighborhood.


## 7. Xem mot vai prediction kem metadata

In [11]:
test_metadata_df = metadata_df.iloc[test_idx].copy()
test_metadata_df["pred_proba_pathogenic"] = proba_test
test_metadata_df["pred_label"] = pred_test

display(
    test_metadata_df[[
        "GeneSymbol",
        "ClinicalSignificance",
        "Y",
        "pred_proba_pathogenic",
        "pred_label",
        "Chromosome",
        "PositionVCF",
        "ReferenceAlleleVCF",
        "AlternateAlleleVCF",
        "ReviewStatus",
    ]]
    .sort_values("pred_proba_pathogenic", ascending=False)
    .head(20)
)

,GeneSymbol,ClinicalSignificance,Y,pred_proba_pathogenic,pred_label,Chromosome,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,ReviewStatus
460220,ASPH,Likely pathogenic,1,0.994565,1,8,61548070,C,G,"criteria provided, single submitter"
291690,RPS6KA3,Likely pathogenic,1,0.994511,1,X,20195064,C,A,"criteria provided, single submitter"
268389,NDUFAF5,Likely pathogenic,1,0.994022,1,20,13787353,G,C,"criteria provided, single submitter"
54761,CDKL5,Pathogenic,1,0.991158,1,X,18575491,G,T,"criteria provided, single submitter"
433459,TRIM63,Likely pathogenic,1,0.991009,1,1,26067335,C,T,"criteria provided, single submitter"
101080,CDKL5,Pathogenic,1,0.990762,1,X,18507161,G,C,"criteria provided, single submitter"
301195,AGL,Likely pathogenic,1,0.989260,1,1,99888109,G,C,"criteria provided, single submitter"
235176,RB1,Pathogenic,1,0.988741,1,13,48381444,G,T,"criteria provided, single submitter"
398969,VPS13B,Likely pathogenic,1,0.988605,1,8,99013936,G,C,"criteria provided, single submitter"
105298,EYA1,Pathogenic,1,0.987709,1,8,71244602,C,A,"criteria provided, single submitter"


## 8. Luu model positional encoding, predictions, history va threshold table


In [12]:
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "clinvar_cnn_gene_group_positional_encoding_pytorch.pt"
prediction_path = PROCESSED_DIR / "cnn_gene_group_positional_encoding_test_predictions_pytorch.parquet"
history_path = PROCESSED_DIR / "cnn_gene_group_positional_encoding_training_history_pytorch.parquet"
threshold_path = PROCESSED_DIR / "cnn_gene_group_positional_encoding_threshold_table_pytorch.parquet"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_class": "ClinVarCNN",
        "input_shape": (INPUT_CHANNELS, 201),
        "base_channels": ["ref_A", "ref_C", "ref_G", "ref_T", "alt_A", "alt_C", "alt_G", "alt_T", "mutation_marker"],
        "positional_encoding_dim": POSITIONAL_ENCODING_DIM,
        "positional_encoding_type": "sinusoidal_absolute_window_position",
        "threshold": 0.5,
        "split_mode": SPLIT_MODE,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "pos_weight_mode": POS_WEIGHT_MODE,
        "use_hard_negative_finetune": USE_HARD_NEGATIVE_FINETUNE,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "test_metrics": test_metrics,
    },
    model_path,
)

test_metadata_df.to_parquet(prediction_path, index=False, engine="pyarrow")
history_df.to_parquet(history_path, index=False, engine="pyarrow")
threshold_df.to_parquet(threshold_path, index=False, engine="pyarrow")

print(f"Saved model: {model_path}")
print(f"Saved predictions: {prediction_path}")
print(f"Saved history: {history_path}")
print(f"Saved threshold table: {threshold_path}")


Saved model: /mnt/MyData/Bioinformatics/Project/models/clinvar_cnn_gene_group_positional_encoding_pytorch.pt
Saved predictions: /mnt/MyData/Bioinformatics/Project/processed/cnn_gene_group_positional_encoding_test_predictions_pytorch.parquet
Saved history: /mnt/MyData/Bioinformatics/Project/processed/cnn_gene_group_positional_encoding_training_history_pytorch.parquet
Saved threshold table: /mnt/MyData/Bioinformatics/Project/processed/cnn_gene_group_positional_encoding_threshold_table_pytorch.parquet
